# Module 08: Interactive Pytest, Fixtures & Quality Assurance

### What You Will Discover
By running this notebook, you will explore how `pytest` fixtures inject dependencies, how parameterization runs test matrices without duplicated code, and how mocks isolate external systems.

**Key Question Answered:** *Why does patching an object at its definition site fail to mock it when another module imports it directly?*


In [ ]:
# Step 1: Plain assertion vs Pytest introspection
def calculate_discount(price: float, discount: float) -> float:
    return price * (1.0 - discount)

assert calculate_discount(100.0, 0.2) == 80.0
print('Basic assert passed!')


In [ ]:
# Step 2: Floating point precision challenge
price = 0.1 + 0.2
print(f'0.1 + 0.2 evaluates to: {price}')
print(f'Is price == 0.3? {price == 0.3}')


In [ ]:
# Step 3: Pytest approx for safe float comparisons
import pytest

assert price == pytest.approx(0.3, rel=1e-6)
print('pytest.approx passed cleanly!')


### 🔮 Prediction Prompt
**Before running the next cell:** In `unittest.mock.patch`, if `worker.py` contains `from service import send_email`, which target string should you patch? `service.send_email` or `worker.send_email`? Write down your choice.


In [ ]:
# Surprising Result: The Golden Rule of Mocking (Patch where it's used)
from unittest.mock import MagicMock


class ExternalAPI:
    def charge(self, amount: int): return 'real_charge'

mock_api = MagicMock()
mock_api.charge.return_value = 'mocked_ok'

print(f'Mock return value: {mock_api.charge(500)}')
mock_api.charge.assert_called_once_with(500)
print('Explanation: Mock verified the call argument and return value successfully!')


### Pytest Fixture Lifecycle & Dependency Injection
Fixtures with `yield` execute cleanup code after the test completes, even if the test fails.


In [ ]:
def test_order_pipeline():
    # Simulating fixture setup
    state = {'orders': []}
    try:
        state['orders'].append('ord_101')
        assert len(state['orders']) == 1
    finally:
        # Simulating fixture teardown
        state['orders'].clear()
        assert len(state['orders']) == 0

test_order_pipeline()
print('Simulated fixture lifecycle completed.')


### Autospec Protection Against Silently Passing Mocks
Unspecced mocks allow calling non-existent methods (`mock.chagre()` instead of `mock.charge()`) and pass tests falsely.


In [ ]:
from unittest.mock import create_autospec


class PaymentClient:
    def send_payment(self, amount: int): pass

safe_mock = create_autospec(PaymentClient)
try:
    # Attempting to call misspelled method raises AttributeError!
    safe_mock.send_paymnet(100)
except AttributeError as exc:
    print(f'create_autospec caught typo immediately: {exc}')


### 🛠️ Interactive Challenge: Fix the Floating Point Assertion Error
The following test function fails due to IEEE 754 floating-point inaccuracies. Fix the assertion using `pytest.approx`.


In [ ]:
# TODO: FIX ME - Use pytest.approx so floating-point inaccuracy does not fail the test
def test_currency_conversion():
    usd = 100.0
    eur_rate = 0.9234
    fee = 1.05
    actual = (usd * eur_rate) + fee
    expected = 93.39  # 93.39000000000001
    # FIX: assert actual == pytest.approx(expected, rel=1e-4)
    assert actual == pytest.approx(expected, rel=1e-4)
    return 'Test passed successfully!'

print(test_currency_conversion())


### 🏁 Summary & Next Steps
- Use `pytest.approx` for all floating point assertions.
- Always patch objects where they are looked up, not where defined.
- Use `autospec=True` on mocks to prevent typo-driven false passes.
- Run `pytest -v 01_pytest_basics_and_fixtures_demo.py` and `02_mocking_and_patching_demo.py`.
- Complete the project in [PROJECT_GUIDE.md](PROJECT_GUIDE.md).
